# SQL 聚合与分组 (Aggregation & Grouping)

> **适用场景**: 多维度数据汇总、报表生成、执行计划理解
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录

1. SQL 执行顺序（理解一切的基础）
2. HAVING vs WHERE 执行顺序
3. GROUP BY ROLLUP — 自动生成小计
4. GROUP BY CUBE — 全维度交叉汇总
5. GROUP BY GROUPING SETS — 自定义汇总组合
6. FILTER (WHERE ...) — 条件聚合
7. DISTINCT vs GROUP BY 性能差异
8. 练习题

In [ ]:
# !pip install duckdb -q
import duckdb
import pandas as pd

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

In [ ]:
# 创建示例数据：零售销售数据（多维度）
con.execute("""
CREATE OR REPLACE TABLE retail_sales AS
SELECT * FROM (VALUES
    ('2024-Q1', '华北', '电子产品', 'A', 15000.0),
    ('2024-Q1', '华北', '服装',     'B', 8000.0),
    ('2024-Q1', '华南', '电子产品', 'A', 22000.0),
    ('2024-Q1', '华南', '服装',     'C', 5500.0),
    ('2024-Q1', '华东', '电子产品', 'B', 18000.0),
    ('2024-Q1', '华东', '食品',     'A', 3200.0),
    ('2024-Q2', '华北', '电子产品', 'A', 17500.0),
    ('2024-Q2', '华北', '服装',     'B', 9200.0),
    ('2024-Q2', '华南', '电子产品', 'C', 25000.0),
    ('2024-Q2', '华南', '食品',     'A', 4100.0),
    ('2024-Q2', '华东', '服装',     'B', 7800.0),
    ('2024-Q2', '华东', '食品',     'C', 2900.0),
    ('2024-Q3', '华北', '食品',     'A', 3800.0),
    ('2024-Q3', '华南', '电子产品', 'B', 28000.0),
    ('2024-Q3', '华东', '服装',     'A', 11000.0)
) t(quarter, region, category, channel, revenue)
""")

# 员工绩效数据
con.execute("""
CREATE OR REPLACE TABLE emp_performance AS
SELECT * FROM (VALUES
    ('Alice',   'Engineering', 'Senior',   95000, 4.5),
    ('Bob',     'Engineering', 'Junior',   72000, 3.8),
    ('Carol',   'Engineering', 'Senior',  102000, 4.9),
    ('David',   'Marketing',   'Senior',   85000, 4.2),
    ('Eve',     'Marketing',   'Junior',   65000, 3.5),
    ('Frank',   'Marketing',   'Junior',   68000, 3.9),
    ('Grace',   'HR',          'Senior',   75000, 4.1),
    ('Heidi',   'HR',          'Junior',   58000, 3.7),
    ('Ivan',    'Engineering', 'Mid',      88000, 4.3),
    ('Judy',    'Marketing',   'Senior',   92000, 4.7)
) t(name, department, level, salary, rating)
""")

print("示例数据创建完成！")
print("\n--- retail_sales 表 ---")
con.execute("SELECT * FROM retail_sales").df()

---

## 1. SQL 执行顺序（理解一切的基础）

SQL 的**书写顺序**与**执行顺序**不同！这是理解 HAVING vs WHERE、列别名使用限制等问题的根本。

```
书写顺序:                    执行顺序:
1. SELECT                   1. FROM / JOIN
2. FROM                     2. WHERE        ← 过滤原始行
3. WHERE                    3. GROUP BY     ← 分组
4. GROUP BY                 4. 聚合函数     ← 计算聚合值
5. HAVING                   5. HAVING       ← 过滤聚合后的组
6. ORDER BY                 6. SELECT       ← 选择列
7. LIMIT                    7. DISTINCT     ← 去重
                            8. ORDER BY     ← 排序
                            9. LIMIT        ← 限制行数
```

**关键推论**:
- WHERE 中**不能**使用聚合函数（因为 WHERE 在 GROUP BY 之前执行）
- HAVING 中**不能**使用 SELECT 中定义的别名（在某些数据库中，如标准 SQL）
- ORDER BY 中**可以**使用 SELECT 中定义的别名（ORDER BY 最后执行）
- 窗口函数在 SELECT 阶段执行，因此不能在 WHERE 或 HAVING 中直接使用

---

## 2. HAVING vs WHERE 执行顺序

| 对比 | WHERE | HAVING |
|------|-------|--------|
| 执行时机 | GROUP BY **之前** | GROUP BY **之后** |
| 过滤对象 | 原始行 | 聚合后的组 |
| 能否用聚合函数 | 不能 | 能 |
| 性能 | 更高（先过滤再聚合）| 较低（先聚合再过滤）|

**优化原则**: 能用 WHERE 就不用 HAVING，先过滤原始数据，减少 GROUP BY 的处理量。

In [ ]:
# WHERE vs HAVING 功能对比

# 示例 1：WHERE 过滤原始行，HAVING 过滤聚合结果
result = con.execute("""
SELECT
    region,
    category,
    SUM(revenue) AS total_revenue,
    COUNT(*)     AS record_count
FROM retail_sales
WHERE quarter IN ('2024-Q1', '2024-Q2')  -- 先过滤：只看前两季度
GROUP BY region, category
HAVING SUM(revenue) > 10000              -- 后过滤：只保留总收入 > 10000 的组
ORDER BY total_revenue DESC
""").df()

print("WHERE (过滤原始行) + HAVING (过滤聚合结果) 组合使用:")
result

In [ ]:
# 常见错误示例：不能在 WHERE 中使用聚合函数
print("错误示例（会报错）:")
print("""
SELECT region, SUM(revenue)
FROM retail_sales
WHERE SUM(revenue) > 10000  -- ❌ 错误：WHERE 不能用聚合函数
GROUP BY region
""")

print("正确写法:")
result = con.execute("""
SELECT region, SUM(revenue) AS total_revenue
FROM retail_sales
GROUP BY region
HAVING SUM(revenue) > 20000  -- 正确：HAVING 可以用聚合函数
ORDER BY total_revenue DESC
""").df()
result

In [ ]:
# 性能优化：WHERE 先过滤，HAVING 后过滤
# 两个查询语义相同，但性能不同

# 低效写法（无 WHERE 过滤，让所有行参与 GROUP BY）
print("低效写法：先 GROUP BY 所有行，再 HAVING 过滤")
result1 = con.execute("""
EXPLAIN
SELECT category, SUM(revenue)
FROM retail_sales
GROUP BY category
HAVING category != '食品'
""").fetchall()
for row in result1:
    print(row[0])

print("\n高效写法：WHERE 先过滤，减少 GROUP BY 处理量")
result2 = con.execute("""
EXPLAIN
SELECT category, SUM(revenue)
FROM retail_sales
WHERE category != '食品'  -- 提前过滤
GROUP BY category
""").fetchall()
for row in result2:
    print(row[0])

---

## 3. GROUP BY ROLLUP — 自动生成小计和总计

`ROLLUP(col1, col2, ...)` 自动生成以下分组级别（从右到左递减）：

```
ROLLUP(A, B, C) 等价于 GROUPING SETS:
  (A, B, C)  -- 最细粒度
  (A, B)     -- 省略 C
  (A)        -- 省略 B, C
  ()         -- 总计行（全部 NULL）
```

**典型用途**: 生成带有小计/合计行的报表，如财务汇报表。

In [ ]:
# ROLLUP 演示：区域 -> 类别 -> 渠道 三级小计
result = con.execute("""
SELECT
    COALESCE(region,   '【所有区域】') AS region,
    COALESCE(category, '【所有类别】') AS category,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(*)               AS record_count,
    -- GROUPING() 函数：该列为 ROLLUP 汇总时返回 1，否则返回 0
    GROUPING(region)   AS is_region_rollup,
    GROUPING(category) AS is_category_rollup
FROM retail_sales
GROUP BY ROLLUP(region, category)
ORDER BY
    GROUPING(region),      -- 汇总行排在后面
    GROUPING(category),
    region,
    category
""").df()

print("ROLLUP 结果（自动生成小计和总计行）:")
result

---

## 4. GROUP BY CUBE — 全维度交叉汇总

`CUBE(col1, col2, ...)` 生成**所有可能的维度组合**（2^n 个分组）：

```
CUBE(A, B) 等价于 GROUPING SETS:
  (A, B)  -- 两维组合
  (A)     -- 只按 A
  (B)     -- 只按 B
  ()      -- 总计
```

**典型用途**: 多维 OLAP 分析、数据立方体、透视表报告。

**注意**: 维度越多，CUBE 生成的分组数越多（性能消耗大），实际使用时要权衡。

In [ ]:
# CUBE 演示：区域 x 类别 全维度交叉
result = con.execute("""
SELECT
    COALESCE(region,   '【全部区域】') AS region,
    COALESCE(category, '【全部类别】') AS category,
    ROUND(SUM(revenue), 2) AS total_revenue,
    GROUPING(region)   AS grp_region,
    GROUPING(category) AS grp_category
FROM retail_sales
GROUP BY CUBE(region, category)
ORDER BY
    grp_region,
    grp_category,
    region,
    category
""").df()

print(f"CUBE 结果共 {len(result)} 行（2^2=4 种分组 × 具体值）:")
print("包含: 区域×类别 / 只按区域 / 只按类别 / 总计")
result

---

## 5. GROUP BY GROUPING SETS — 自定义汇总组合

`GROUPING SETS` 允许你**精确指定**需要哪些分组组合，避免 CUBE 的冗余计算。

```sql
GROUP BY GROUPING SETS (
    (A, B),   -- 按 A+B 分组
    (A),      -- 只按 A 分组
    ()        -- 总计
)
```

**ROLLUP vs CUBE vs GROUPING SETS 对比**:
- `ROLLUP(A,B)` = `GROUPING SETS((A,B), (A), ())`
- `CUBE(A,B)` = `GROUPING SETS((A,B), (A), (B), ())`
- `GROUPING SETS` 最灵活，完全自定义

In [ ]:
# GROUPING SETS：只生成业务需要的分组组合
result = con.execute("""
SELECT
    COALESCE(region,   '【汇总】') AS region,
    COALESCE(category, '【汇总】') AS category,
    COALESCE(quarter,  '【汇总】') AS quarter,
    ROUND(SUM(revenue), 2) AS total_revenue,
    -- 标识当前是哪种分组
    CASE
        WHEN GROUPING(region)=0 AND GROUPING(category)=0 THEN '区域×类别'
        WHEN GROUPING(region)=0 AND GROUPING(quarter)=0  THEN '区域×季度'
        WHEN GROUPING(region)=0                           THEN '区域小计'
        ELSE                                                   '总计'
    END AS grouping_level
FROM retail_sales
GROUP BY GROUPING SETS (
    (region, category),  -- 我们需要：区域×类别
    (region, quarter),   -- 我们需要：区域×季度
    (region),            -- 我们需要：区域小计
    ()                   -- 总计
)
ORDER BY grouping_level, region, category, quarter
""").df()

print("GROUPING SETS 自定义汇总组合:")
result

---

## 6. FILTER (WHERE ...) — 条件聚合

`FILTER` 子句允许对聚合函数添加条件过滤，**在同一个查询中实现多个条件的聚合**。

```sql
SUM(revenue) FILTER (WHERE category = '电子产品') AS electronics_revenue
```

**优势**: 比使用 CASE WHEN 更清晰，且通常性能更好（优化器更容易优化）

**等价写法对比**:
```sql
-- FILTER 写法（推荐）:
SUM(salary) FILTER (WHERE level = 'Senior') AS senior_total

-- CASE WHEN 写法（等价，但冗长）:
SUM(CASE WHEN level = 'Senior' THEN salary ELSE 0 END) AS senior_total
```

In [ ]:
# FILTER 子句：同一查询中多条件聚合
result = con.execute("""
SELECT
    department,
    COUNT(*)                                          AS total_employees,
    -- 各级别人数
    COUNT(*) FILTER (WHERE level = 'Senior')          AS senior_count,
    COUNT(*) FILTER (WHERE level = 'Mid')             AS mid_count,
    COUNT(*) FILTER (WHERE level = 'Junior')          AS junior_count,
    -- 各级别平均薪资
    ROUND(AVG(salary) FILTER (WHERE level = 'Senior'), 0) AS avg_senior_salary,
    ROUND(AVG(salary) FILTER (WHERE level = 'Junior'), 0) AS avg_junior_salary,
    -- 高绩效（rating >= 4.5）员工数量
    COUNT(*) FILTER (WHERE rating >= 4.5)             AS high_performer_count,
    -- 高绩效员工薪资总和
    SUM(salary) FILTER (WHERE rating >= 4.5)          AS high_performer_salary_sum
FROM emp_performance
GROUP BY department
ORDER BY department
""").df()

print("使用 FILTER 实现多条件聚合（一次扫描完成）:")
result

In [ ]:
# FILTER 与 CASE WHEN 对比
result = con.execute("""
SELECT
    department,
    -- FILTER 写法（简洁）
    SUM(salary) FILTER (WHERE level = 'Senior') AS senior_salary_filter,
    -- CASE WHEN 等价写法
    SUM(CASE WHEN level = 'Senior' THEN salary ELSE 0 END) AS senior_salary_case,
    -- 验证两者结果相同
    SUM(salary) FILTER (WHERE level = 'Senior') =
    SUM(CASE WHEN level = 'Senior' THEN salary ELSE 0 END) AS results_match
FROM emp_performance
GROUP BY department
""").df()

print("FILTER vs CASE WHEN 结果对比（结果完全相同）:")
result

---

## 7. DISTINCT vs GROUP BY 性能差异

在只需要去重（不需要聚合函数）时，`DISTINCT` 和 `GROUP BY` 产生相同结果，但性能可能不同。

### 语义对比

```sql
-- 两者等价（无聚合函数时）:
SELECT DISTINCT region, category FROM retail_sales;
SELECT region, category FROM retail_sales GROUP BY region, category;
```

### 性能规律（因数据库而异）

| 场景 | 推荐 | 原因 |
|------|------|------|
| 纯去重，无聚合 | `DISTINCT` | 语义更清晰，优化器通常同等效率 |
| 需要聚合函数 | `GROUP BY` | DISTINCT 不支持 SUM/COUNT |
| 子查询中检查存在性 | `EXISTS` | 找到第一条即停止，效率更高 |
| 大表 JOIN 后去重 | `GROUP BY` | 优化器有更多优化选项 |

**面试关键**: 现代优化器通常将两者优化为相同执行计划，关键在于**语义清晰**和**避免不必要的去重**（如已知列唯一则不需要 DISTINCT）。

In [ ]:
# DISTINCT vs GROUP BY 执行计划对比
print("=== DISTINCT 执行计划 ===")
plan1 = con.execute("""
EXPLAIN SELECT DISTINCT region, category FROM retail_sales
""").fetchall()
for row in plan1:
    print(row[0])

print("\n=== GROUP BY 执行计划 ===")
plan2 = con.execute("""
EXPLAIN SELECT region, category FROM retail_sales GROUP BY region, category
""").fetchall()
for row in plan2:
    print(row[0])

In [ ]:
# COUNT(DISTINCT) vs COUNT(*) 的区别
result = con.execute("""
SELECT
    quarter,
    COUNT(*)                AS total_records,         -- 总记录数
    COUNT(DISTINCT region)  AS unique_regions,        -- 出现的唯一区域数
    COUNT(DISTINCT category) AS unique_categories,    -- 出现的唯一类别数
    SUM(revenue)            AS total_revenue,
    -- 每个唯一区域的平均收入
    ROUND(SUM(revenue) / COUNT(DISTINCT region), 2) AS avg_revenue_per_region
FROM retail_sales
GROUP BY quarter
ORDER BY quarter
""").df()

print("COUNT(*) vs COUNT(DISTINCT col) 示例:")
result

---

## 练习题

以下练习涵盖真实业务中的多维汇总场景。

In [ ]:
# 练习数据准备
con.execute("""
CREATE OR REPLACE TABLE transactions AS
SELECT * FROM (VALUES
    (1,  'C001', '2024-01', 'Mobile',  'Electronics', 1500.0),
    (2,  'C001', '2024-01', 'Desktop', 'Books',         80.0),
    (3,  'C002', '2024-01', 'Mobile',  'Clothing',      350.0),
    (4,  'C003', '2024-01', 'Mobile',  'Electronics',  2200.0),
    (5,  'C001', '2024-02', 'Mobile',  'Electronics',   950.0),
    (6,  'C002', '2024-02', 'Desktop', 'Electronics',  1800.0),
    (7,  'C002', '2024-02', 'Mobile',  'Books',          45.0),
    (8,  'C003', '2024-02', 'Desktop', 'Clothing',      420.0),
    (9,  'C001', '2024-03', 'Desktop', 'Books',         120.0),
    (10, 'C003', '2024-03', 'Mobile',  'Electronics',  3100.0),
    (11, 'C002', '2024-03', 'Mobile',  'Clothing',      280.0),
    (12, 'C003', '2024-03', 'Desktop', 'Books',          95.0)
) t(txn_id, customer_id, month, platform, category, amount)
""")
print("练习数据准备完成！")
con.execute("SELECT * FROM transactions ORDER BY txn_id").df()

### 练习 1: 多维汇总报表（ROLLUP）

**需求**: 生成一份销售汇总报表，显示：
- 每个月每个类别的总销售额（最细粒度）
- 每个月的小计
- 全部月份的总计

使用 `COALESCE` 将 NULL 替换为 '小计' / '总计'，并用 `GROUPING()` 标识汇总行。

In [ ]:
# 练习 1: ROLLUP 多维汇总（填写 TODO）
result = con.execute("""
SELECT
    COALESCE(month,    '总计')  AS month,
    COALESCE(category, '小计')  AS category,
    ROUND(SUM(amount), 2)        AS total_amount,
    COUNT(*)                     AS num_transactions,
    -- TODO: 添加 GROUPING() 标识
    GROUPING(TODO) AS is_month_subtotal
FROM transactions
-- TODO: 使用 ROLLUP 按 month, category 分组
GROUP BY TODO
ORDER BY
    GROUPING(month),
    GROUPING(category),
    month,
    category
""").df()
result

In [ ]:
# 练习 1 参考答案
result = con.execute("""
SELECT
    COALESCE(month,    '总计') AS month,
    COALESCE(category, '小计') AS category,
    ROUND(SUM(amount), 2)       AS total_amount,
    COUNT(*)                    AS num_transactions,
    GROUPING(month)             AS is_month_subtotal
FROM transactions
GROUP BY ROLLUP(month, category)
ORDER BY
    GROUPING(month),
    GROUPING(category),
    month,
    category
""").df()
result

### 练习 2: 条件聚合分析（FILTER）

**需求**: 对每个月生成一份平台对比报告：
1. Mobile 平台的总销售额
2. Desktop 平台的总销售额
3. Mobile vs Desktop 的销售额比例（Mobile / Desktop）
4. Electronics 类别的总销售额
5. 非 Electronics 类别的总销售额

**要求**: 使用 `FILTER` 子句实现，不使用多个子查询。

In [ ]:
# 练习 2: 条件聚合分析（填写 TODO）
result = con.execute("""
SELECT
    month,
    -- TODO 1: Mobile 平台销售额
    SUM(amount) FILTER (WHERE TODO) AS mobile_revenue,
    -- TODO 2: Desktop 平台销售额
    SUM(amount) FILTER (WHERE TODO) AS desktop_revenue,
    -- TODO 3: Mobile/Desktop 比率
    ROUND(
        SUM(amount) FILTER (WHERE platform = 'Mobile') /
        NULLIF(SUM(amount) FILTER (WHERE platform = 'Desktop'), 0),
        2
    ) AS mobile_to_desktop_ratio,
    -- TODO 4: Electronics 销售额
    SUM(amount) FILTER (WHERE TODO) AS electronics_revenue,
    -- TODO 5: 非 Electronics 销售额
    SUM(amount) FILTER (WHERE TODO) AS non_electronics_revenue
FROM transactions
GROUP BY month
ORDER BY month
""").df()
result

In [ ]:
# 练习 2 参考答案
result = con.execute("""
SELECT
    month,
    SUM(amount) FILTER (WHERE platform = 'Mobile')          AS mobile_revenue,
    SUM(amount) FILTER (WHERE platform = 'Desktop')         AS desktop_revenue,
    ROUND(
        SUM(amount) FILTER (WHERE platform = 'Mobile') /
        NULLIF(SUM(amount) FILTER (WHERE platform = 'Desktop'), 0),
        2
    )                                                        AS mobile_to_desktop_ratio,
    SUM(amount) FILTER (WHERE category = 'Electronics')     AS electronics_revenue,
    SUM(amount) FILTER (WHERE category != 'Electronics')    AS non_electronics_revenue
FROM transactions
GROUP BY month
ORDER BY month
""").df()
result

### 练习 3: GROUPING SETS 自定义报表

**需求**: 业务需要一份包含以下维度组合的汇总报表：
1. 按 `platform × category` 的销售额
2. 按 `platform` 的销售小计
3. 按 `category` 的销售小计
4. 总计

**要求**: 使用 `GROUPING SETS`（不要用 CUBE，避免生成 platform × month 等不需要的组合）

In [ ]:
# 练习 3: GROUPING SETS（填写 TODO）
result = con.execute("""
SELECT
    COALESCE(platform, '全平台') AS platform,
    COALESCE(category, '全类别') AS category,
    ROUND(SUM(amount), 2)         AS total_amount,
    COUNT(DISTINCT customer_id)   AS unique_customers,
    CASE
        WHEN GROUPING(platform) = 0 AND GROUPING(category) = 0 THEN '平台×类别'
        WHEN GROUPING(platform) = 0                             THEN '平台小计'
        WHEN GROUPING(category) = 0                             THEN '类别小计'
        ELSE                                                          '总计'
    END AS grouping_desc
FROM transactions
-- TODO: 使用 GROUPING SETS 指定上述4种分组
GROUP BY GROUPING SETS (TODO)
ORDER BY grouping_desc DESC, platform, category
""").df()
result

In [ ]:
# 练习 3 参考答案
result = con.execute("""
SELECT
    COALESCE(platform, '全平台') AS platform,
    COALESCE(category, '全类别') AS category,
    ROUND(SUM(amount), 2)         AS total_amount,
    COUNT(DISTINCT customer_id)   AS unique_customers,
    CASE
        WHEN GROUPING(platform) = 0 AND GROUPING(category) = 0 THEN '平台×类别'
        WHEN GROUPING(platform) = 0                             THEN '平台小计'
        WHEN GROUPING(category) = 0                             THEN '类别小计'
        ELSE                                                          '总计'
    END AS grouping_desc
FROM transactions
GROUP BY GROUPING SETS (
    (platform, category),
    (platform),
    (category),
    ()
)
ORDER BY grouping_desc DESC, platform, category
""").df()
result

### 练习 4: 执行顺序推理题

**场景**: 分析以下 SQL 并回答问题。

```sql
SELECT
    platform,
    SUM(amount) AS total,     -- 别名 'total'
    COUNT(*) AS cnt
FROM transactions
WHERE amount > 100
GROUP BY platform
HAVING SUM(amount) > 2000    -- 注意：某些数据库能用别名，某些不能
ORDER BY total DESC           -- 用了别名
LIMIT 5
```

**问题**:
1. WHERE 和 HAVING 分别过滤了什么？
2. 为什么 HAVING 要写 `SUM(amount)` 而不能用 `WHERE SUM(amount) > 2000`？
3. ORDER BY 中可以用别名 `total`，为什么？

**请在下方代码格中验证你的理解：**

In [ ]:
# 练习 4: 执行顺序验证
# 步骤 1: 查看 WHERE 过滤后的原始数据
print("步骤 1: WHERE amount > 100 之后的数据:")
display(con.execute("""
SELECT * FROM transactions WHERE amount > 100 ORDER BY platform, amount
""").df())

# 步骤 2: GROUP BY 之后的数据
print("\n步骤 2: GROUP BY platform 之后:")
display(con.execute("""
SELECT platform, SUM(amount) AS total, COUNT(*) AS cnt
FROM transactions
WHERE amount > 100
GROUP BY platform
""").df())

# 步骤 3: HAVING 过滤后
print("\n步骤 3: HAVING SUM(amount) > 2000 之后:")
display(con.execute("""
SELECT platform, SUM(amount) AS total, COUNT(*) AS cnt
FROM transactions
WHERE amount > 100
GROUP BY platform
HAVING SUM(amount) > 2000
ORDER BY total DESC
""").df())

---

## 复习要点

### SQL 执行顺序速记
```
FROM → WHERE → GROUP BY → 聚合 → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT
```

### ROLLUP / CUBE / GROUPING SETS 选择指南

| 需求 | 选择 |
|------|------|
| 需要层级小计（A, A+B, A+B+C）| `ROLLUP` |
| 需要所有维度组合 | `CUBE` |
| 需要特定几种组合 | `GROUPING SETS` |

### 条件聚合最佳实践

```sql
-- 推荐：FILTER 子句（更清晰）
SUM(amount) FILTER (WHERE category = 'A')  AS cat_a_sum

-- 等价：CASE WHEN（兼容性更好，但冗长）
SUM(CASE WHEN category = 'A' THEN amount ELSE 0 END)  AS cat_a_sum
```

### 面试高频问题

1. "WHERE 和 HAVING 的区别" → 执行顺序：WHERE 在 GROUP BY 前，HAVING 在 GROUP BY 后
2. "如何生成带小计的报表" → ROLLUP 或 GROUPING SETS
3. "如何在一次查询中统计不同条件的数量" → COUNT(*) FILTER (WHERE ...)
4. "DISTINCT 和 GROUP BY 哪个快" → 通常等效，关键是避免不必要的去重操作